# 🤖 AI Data Copilot — Análise de Churn (Telecom)

> Agent de IA com LLM + LangChain para geração automática de insights sobre churn de clientes no setor de telecomunicações.

---

## 🚀 Sobre o Projeto

Este notebook faz parte de um projeto maior que combina **Dashboard interativo** + **AI Data Copilot** para análise de churn.
O objetivo é demonstrar na prática como integrar **BI + IA** para gerar insights automáticos a partir de dados reais.

---

## 🔗 Links do Projeto

| | |
|---|---|
| 📊 **Dashboard (Railway)** | [workspacechurn-dashboard-production.up.railway.app](https://workspacechurn-dashboard-production.up.railway.app/) |
| 🛠️ **Versão inicial (Replit)** | [Acessar Replit](https://b90be4c4-970a-4140-ab69-8459bfb8d618-00-3h6b71i09oqlk.janeway.replit.dev/) |
| 👤 **LinkedIn** | [lucas-diagone](https://www.linkedin.com/in/lucas-diagone-691285104/) |

---

## 📂 Fonte de Dados

Dataset público de churn (Telecom):

| | |
|---|---|
| 🔗 **Kaggle** | [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) |
| 🔗 **Google Sheets (fonte ativa)** | [Acessar planilha](https://docs.google.com/spreadsheets/d/1W9_d6wi9x7CB5z0Rx0km7LmiOvfW_laW2ywFSAy2QRU/export?format=csv) |

> 📌 **Por que Google Sheets?** Optei por utilizar o Google Sheets como fonte de dados para permitir atualizações contínuas da base, possibilitando que o projeto reflita automaticamente novas informações sem necessidade de reprocessamento manual.

---

## ⚙️ Passo a Passo — Configuração do Ambiente

### 1️⃣ Crie um ambiente conda isolado (Anaconda Prompt):

```bash
conda create -n churn_agent python=3.11 -y
conda activate churn_agent
```

> ⚠️ Certifique-se que o prompt mostra `(churn_agent)` antes de continuar.

### 2️⃣ Instale o NumPy via conda (evita erro de compilador no Windows):

```bash
conda install numpy -y
```

### 3️⃣ Remova pacotes conflitantes (se existirem):

```bash
pip uninstall langchain-classic langgraph-prebuilt -y
```

### 4️⃣ Instale as dependências com versões fixas:

```bash
pip install langchain==0.2.16 langchain-core==0.2.38 langchain-community==0.2.16 langchain-experimental==0.0.65 langchain-groq pandas jupyter chardet
```

### 5️⃣ Abra o Jupyter pelo Anaconda Prompt:

```bash
jupyter notebook
```

> ⚠️ No Jupyter, selecione o kernel correto: **Kernel → Change Kernel → churn_agent**
> Se o kernel não aparecer, rode:
> ```bash
> python -m ipykernel install --user --name=churn_agent --display-name "churn_agent"
> ```

---

## 🔑 API Key — Groq (gratuito)

1. Acesse [console.groq.com](https://console.groq.com)
2. Crie uma conta gratuita (sem cartão de crédito)
3. Gere uma API Key — ela começa com `gsk_...`
4. Cole sua chave na **Célula 4** do notebook

---

## 📦 Versões Utilizadas

| Pacote | Versão |
|---|---|
| Python | 3.11 |
| langchain | 0.2.16 |
| langchain-core | 0.2.38 |
| langchain-community | 0.2.16 |
| langchain-experimental | 0.0.65 |
| langchain-groq | latest |
| LLM | llama-3.3-70b-versatile (Groq) |
| pandas | latest |
| numpy | via conda |

## 🔧 0.INSTALAÇÃO DE DEPENDÊNCIAS
Recomendamos seguir o passo a passo do bloco Markdown acima.

Caso prefira instalar direto aqui, rode essa célula uma vez:

ATENÇÃO: No Windows, instale o numpy via Anaconda Prompt primeiro:

conda install numpy -y

In [ ]:
!pip install langchain==0.2.16 langchain-core==0.2.38 langchain-community==0.2.16 langchain-experimental==0.0.65 langchain-groq pandas chardet

## 📦 1. Imports

In [ ]:
import os
import pandas as pd

from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_react_agent
from langchain_experimental.utilities import PythonREPL
from langchain import hub
from langchain_groq import ChatGroq

## 📊 2. Carregamento dos Dados

Os dados são carregados diretamente do **Google Sheets**, garantindo que qualquer atualização na planilha seja refletida automaticamente no projeto.

In [ ]:
def load_data(url):
    df = pd.read_csv(url)
    print("\n📊 Colunas disponíveis:")
    print(df.columns)
    print("\n📊 Preview:")
    print(df.head())
    df["valor_total"] = pd.to_numeric(df["valor_total"], errors="coerce")
    df = df.dropna()
    return df


DATA_URL = "https://docs.google.com/spreadsheets/d/1W9_d6wi9x7CB5z0Rx0km7LmiOvfW_laW2ywFSAy2QRU/export?format=csv"

df = load_data(DATA_URL)

## 🛠️ 3. Python Tool — Análise de Dados

O **Python Agent Tool** permite ao modelo executar código Python diretamente no DataFrame, gerando análises e insights dinâmicos em tempo real, sem precisar de dados pré-processados.

In [ ]:
python_repl = PythonREPL()
python_repl.locals = {"df": df}

python_tool = Tool(
    name="Python Data Analysis",
    func=python_repl.run,
    description="""
Use essa ferramenta para analisar o dataframe df.
O dataframe contém dados de churn.
Você pode calcular métricas, agrupar dados, analisar padrões e usar pandas livremente.
Sempre use essa tool para perguntas sobre dados.
"""
)

tools = [python_tool]

## 🧠 4. LLM — Groq (gratuito, sem download local)

Utilizamos o modelo **llama-3.3-70b-versatile** via [Groq](https://console.groq.com) — um dos modelos mais avançados disponíveis gratuitamente, com excelente desempenho para raciocínio estruturado no formato ReAct.

> 🔑 Substitua `cole_sua_chave_aqui` pela sua API Key do Groq. **Nunca suba a chave para o GitHub!**

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or "cole_sua_chave_aqui"

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

## 📝 5. Prompt Controlado

O prompt instrui o modelo a se comportar como um **analista de dados**, priorizando precisão e nunca inventando números.

In [ ]:
template = """
Você é um analista de dados.

Use ferramentas quando necessário.

Se usar dados:
- Não invente números
- Não arredonde
- Seja preciso

Pergunta:
{input}

{agent_scratchpad}
"""

custom_prompt = PromptTemplate(
    template=template,
    input_variables=["input", "agent_scratchpad"]
)

## 🤖 6. ReAct Agent

O **ReAct Agent** (Reasoning + Acting) combina raciocínio em linguagem natural com execução de ações reais — neste caso, rodar código Python no DataFrame para responder perguntas com dados precisos.

**Fluxo do Agent:**
1. 🤔 Recebe a pergunta
2. 🧠 Raciocina sobre qual ferramenta usar
3. 🛠️ Executa o código Python no DataFrame
4. 📊 Interpreta o resultado
5. 💬 Responde com insights baseados nos dados reais

In [ ]:
react_prompt = hub.pull("hwchase17/react")

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

print("\n🤖 Agent pronto!")

## 💡 7. Demonstração — Exemplos de Perguntas

Abaixo estão exemplos de perguntas que você pode fazer ao AI Copilot. O agent analisa os dados em tempo real e retorna insights precisos:

---

💬 **Pergunta 1:** Qual a taxa geral de churn?
🧠 **Resposta:** A taxa geral de churn é de **26.5%** — ou seja, aproximadamente 1 em cada 4 clientes cancelou o serviço.

---

💬 **Pergunta 2:** Qual a taxa de churn por tipo de contrato?
🧠 **Resposta:** Clientes **Month-to-month** têm a maior taxa de churn (**42.7%**), enquanto contratos anuais têm **11.3%** e bianuais apenas **2.8%**.

---

💬 **Pergunta 3:** Qual o perfil do cliente que mais cancela?
🧠 **Resposta:** Clientes com contrato mensal, que usam fibra ótica, sem suporte técnico e com menos de 12 meses de permanência têm a maior propensão ao churn.

---

💬 **Pergunta 4:** Qual a taxa de churn por método de pagamento?
🧠 **Resposta:** Clientes que pagam via **Electronic Check** têm a maior taxa de churn (**45.3%**), enquanto pagamentos automáticos (cartão ou banco) têm taxas abaixo de **17%**.

---

💬 **Pergunta 5:** Qual a taxa de churn agrupada por faixas de tempo como cliente (tenure)?
🧠 **Resposta:** Clientes com **0-12 meses** têm churn de **47.7%**. Entre **13-24 meses** cai para **28.3%**. Acima de **48 meses** o churn é inferior a **8%**.

---

💬 **Pergunta 6:** Clientes idosos têm maior taxa de churn?
🧠 **Resposta:** Sim. Clientes idosos têm churn de **41.7%**, contra **23.6%** entre não idosos — uma diferença de quase 18 pontos percentuais.

---

💬 **Pergunta 7:** O tipo de internet influencia o churn?
🧠 **Resposta:** Clientes de **Fibra ótica** têm churn de **41.9%**, bem acima dos clientes de **DSL** (**19.0%**) e dos sem internet (**7.4%**).

---

💬 **Pergunta 8:** Qual o valor médio mensal dos clientes que cancelaram vs. os que ficaram?
🧠 **Resposta:** Clientes que cancelaram pagavam em média **R$ 74.44/mês**, enquanto os que permaneceram pagavam **R$ 61.27/mês** — os churners pagavam mais.

---

💬 **Pergunta 9:** Ter suporte técnico reduz o churn?
🧠 **Resposta:** Sim. Clientes **sem suporte técnico** têm churn de **41.7%**, enquanto os **com suporte** têm apenas **15.2%**.

---

💬 **Pergunta 10:** Qual serviço adicional tem maior impacto na retenção?
🧠 **Resposta:** O **suporte técnico** é o serviço com maior impacto na retenção, seguido de **segurança online** e **backup online**. Clientes com esses serviços ativados têm churn até 2x menor.

---

> 💡 O agent irá calcular os valores exatos a partir dos dados reais ao rodar cada pergunta.

## 💬 8. Loop Interativo

Faça suas perguntas em linguagem natural! O agent irá analisar os dados e responder com insights precisos.

Digite `sair` ou `exit` para encerrar.

In [ ]:
if __name__ == "__main__":

    while True:
        question = input("\n💬 Pergunta: ")

        if question.lower() in ["sair", "exit"]:
            print("\n👋 Encerrando o AI Data Copilot. Até mais!")
            break

        response = agent_executor.invoke({
            "input": question
        })

        print("\n🧠 Resposta:\n")
        print(response["output"])